In [2]:
import numpy as np
import open3d as o3d
import laspy as lp
import matplotlib.pyplot as plt

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [3]:
#data loading
dataset  = o3d.data.PLYPointCloud()
pcd = o3d.io.read_point_cloud(dataset.path)

pcd_centre=pcd.get_center()
pcd.translate(-pcd_centre)

PointCloud with 196133 points.

In [4]:
#Preprocessing
nn=16
std_multiplier=10

filtered_pcd=pcd.remove_statistical_outlier(nn,std_multiplier)

outliers=pcd.select_by_index(filtered_pcd[1],invert=True)
outliers.paint_uniform_color([1,0,0])

filtered_pcd=filtered_pcd[0]
o3d.visualization.draw_geometries([filtered_pcd, outliers])

In [5]:
#Voxelization (vectorization)
voxel_size=0.01

pcd_downsampled=filtered_pcd.voxel_down_sample(voxel_size=voxel_size)

o3d.visualization.draw_geometries([pcd_downsampled])

In [6]:
#Estimating Normals
nn_distance=np.mean(pcd.compute_nearest_neighbor_distance())

radius_normals=4*nn_distance

pcd_downsampled.estimate_normals(search_param=o3d.geometry.KDTreeSearchParamHybrid(radius=radius_normals,max_nn=16),fast_normal_computation=True)

pcd_downsampled.paint_uniform_color([0.6,0.6,0.6])
o3d.visualization.draw_geometries([pcd_downsampled,outliers])

In [ ]:
#Environment Setup

pcd=pcd_downsampled
o3d.visualization.draw_geometries([pcd])

In [7]:
#RANSAC planner segmentation

pt_to_plane_dist=0.02

plane_model, inliers=pcd.segment_plane(distance_threshold=pt_to_plane_dist,ransac_n=3,num_iterations=1000)
[a,b,c,d]=plane_model

print(f"Plane equation : {a:2f}x + {b:2f}y + {c:.2f}z + {d:.2f} = 0")

inlier_cloud=pcd.select_by_index(inliers)
outlier_cloud=pcd.select_by_index(inliers,invert=True)

inlier_cloud.paint_uniform_color([0.6,0.6,0.6])
outlier_cloud.paint_uniform_color([1,0,0])
o3d.visualization.draw_geometries([inlier_cloud,outlier_cloud])

Plane equation : 0.000551x + 0.999616y + 0.03z + -0.48 = 0


In [8]:
#Multi-order RANSAC
max_plane_idx=6

segment_models={}
segments={}
rest=pcd

for i in range(max_plane_idx):
    colors=plt.get_cmap("tab20")(i)
    segment_models[i],inliers=rest.segment_plane(distance_threshold=pt_to_plane_dist,ransac_n=3,num_iterations=1000)
    segments[i]=rest.select_by_index(inliers)
    segments[i].paint_uniform_color(list(colors[:3]))
    rest=rest.select_by_index(inliers,invert=True)
    print("pass",i,"/",max_plane_idx,"done.")

o3d.visualization.draw_geometries([segments[i] for i in range(max_plane_idx)]+[rest])

pass 0 / 6 done.
pass 1 / 6 done.
pass 2 / 6 done.
pass 3 / 6 done.
pass 4 / 6 done.
pass 5 / 6 done.


In [9]:
#Eucledian Clustering for the objects

labels= np.array(rest.cluster_dbscan(eps=0.05,min_points=5))
max_label=labels.max()
print(f"point cloud has {max_label+1} clusters")
colors=plt.get_cmap("tab10")(labels/(max_label if max_label>0 else 1))
colors[labels<0]=0
rest.colors=o3d.utility.Vector3dVector(colors[:,:3])
o3d.visualization.draw_geometries([segments[i] for i in range(max_plane_idx)]+[rest])

point cloud has 26 clusters


In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

def create_pointnet_backbone(num_points=1024):
    inputs = keras.Input(shape=(num_points, 3))
    x = layers.Conv1D(64, 1, activation="relu")(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Conv1D(64, 1, activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Conv1D(128, 1, activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Conv1D(1024, 1, activation="relu")(x)
    x = layers.BatchNormalization()(x)
    features = layers.GlobalMaxPooling1D()(x)
    return keras.Model(inputs=inputs, outputs=features, name="pointnet_backbone")

class_names = ["chair", "table", "sofa", "shelf", "decor"]
num_classes = len(class_names)

backbone = create_pointnet_backbone(1024)
backbone.trainable = False

inputs = keras.Input(shape=(1024, 3))
x = backbone(inputs)
x = layers.Dense(256, activation="relu")(x)
x = layers.Dropout(0.2)(x)
x = layers.Dense(128, activation="relu")(x)
outputs = layers.Dense(num_classes, activation="softmax")(x)

detector_model = keras.Model(inputs=inputs, outputs=outputs, name="pointnet_transfer_detector")
detector_model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.003), loss="sparse_categorical_crossentropy", metrics=["accuracy"])


In [ ]:
def generate_prototype(class_id):
    if class_id == 0:
        p = [np.random.uniform([-0.4, -0.4, 0.0], [0.4, 0.4, 0.1], (400, 3)),
             np.random.uniform([-0.4, 0.3, 0.1], [0.4, 0.4, 0.8], (300, 3)),
             np.random.uniform([-0.4, -0.4, -0.6], [0.4, 0.4, 0.0], (324, 3))]
    elif class_id == 1:
        p = [np.random.uniform([-0.9, -0.6, 0.4], [0.9, 0.6, 0.5], (600, 3)),
             np.random.uniform([-0.8, -0.5, -0.5], [0.8, 0.5, 0.4], (424, 3))]
    elif class_id == 2:
        p = [np.random.uniform([-0.8, -0.4, -0.3], [0.8, 0.4, 0.2], (700, 3)),
             np.random.uniform([-0.8, 0.2, 0.2], [0.8, 0.4, 0.6], (324, 3))]
    elif class_id == 3:
        p = [np.random.uniform([-0.4, -0.2, -0.8], [0.4, 0.2, 0.8], (1024, 3))]
    else:
        p = [np.random.uniform([-0.2, -0.2, -0.2], [0.2, 0.2, 0.2], (1024, 3))]
    pts = np.vstack(p)
    pts = pts - np.mean(pts, axis=0)
    scale = np.max(np.sqrt(np.sum(pts**2, axis=1)))
    return pts / scale if scale > 0 else pts

train_x = np.array([generate_prototype(c) for _ in range(20) for c in range(num_classes)])
train_y = np.array([c for _ in range(20) for c in range(num_classes)])
detector_model.fit(train_x, train_y, epochs=15, batch_size=16, verbose=0)

identified_objects = []
bounding_boxes = []
class_palette = [
    [1.0, 0.2, 0.2],
    [0.2, 0.6, 1.0],
    [0.2, 0.8, 0.2],
    [1.0, 0.6, 0.0],
    [0.7, 0.2, 0.9]
]

for cluster_id in range(max_label + 1):
    indices = np.where(labels == cluster_id)[0]
    if len(indices) < 25:
        continue
    cluster_cloud = rest.select_by_index(indices)
    pts = np.asarray(cluster_cloud.points)
    if len(pts) >= 1024:
        sample_idx = np.random.choice(len(pts), 1024, replace=False)
    else:
        sample_idx = np.random.choice(len(pts), 1024, replace=True)
    sampled_pts = pts[sample_idx]
    sampled_pts = sampled_pts - np.mean(sampled_pts, axis=0)
    norm_scale = np.max(np.sqrt(np.sum(sampled_pts**2, axis=1)))
    if norm_scale > 0:
        sampled_pts = sampled_pts / norm_scale
    preds = detector_model.predict(np.expand_dims(sampled_pts, axis=0), verbose=0)
    pred_class_idx = int(np.argmax(preds[0]))
    confidence = float(preds[0][pred_class_idx])
    assigned_color = class_palette[pred_class_idx]
    cluster_cloud.paint_uniform_color(assigned_color)
    bbox = cluster_cloud.get_axis_aligned_bounding_box()
    bbox.color = assigned_color
    identified_objects.append(cluster_cloud)
    bounding_boxes.append(bbox)
    print(f"Object {cluster_id}: {class_names[pred_class_idx]} (confidence: {confidence:.2f})")


In [ ]:
o3d.visualization.draw_geometries([segments[i] for i in range(max_plane_idx)] + identified_objects + bounding_boxes)
